In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import RidgeClassifierCV, LogisticRegression, SGDClassifier, PassiveAggressiveClassifier
# 生成哈德玛矩阵
def hadamard_matrix(n):
    if n == 1:
        return np.array([[1]])
    elif n % 2 != 0 or (n & (n - 1) != 0):
        raise ValueError("哈德玛矩阵的阶数必须是 2 的幂！")
    else:
        H = hadamard_matrix(n // 2)
        return np.block([
            [H, H],
            [H, -H]
        ])

# HITRocketTransform 类
class HITRocketTransform(nn.Module):
    def __init__(self, dilations, bias_quantiles, num_samples=1, hadamard_order=8):
        super(HITRocketTransform, self).__init__()
        self.dilations = dilations
        self.bias_quantiles = torch.tensor(bias_quantiles, dtype=torch.float32)  # 确保是张量
        self.kernel_size = hadamard_order  # 卷积核大小，使用哈德玛矩阵的阶数
        self.num_kernels = hadamard_order  # 卷积核数量，使用哈德玛矩阵的列数
        self.num_samples = num_samples  # 用于计算偏置的样本数量
        
        # 初始化卷积核权重，使用哈德玛矩阵的列
        self.weights = self._initialize_weights(hadamard_order)
        
        # 初始化偏置
        self.biases = None

    def _initialize_weights(self, hadamard_order):
        H = hadamard_matrix(hadamard_order)
        weights = torch.tensor(H.T, dtype=torch.float32)  # 使用哈德玛矩阵的转置作为卷积核权重
        return nn.Parameter(weights.unsqueeze(1), requires_grad=False)

    def _fit_biases(self, x):
        random_sample_indices = np.random.choice(x.shape[0], self.num_samples, replace=False)
        x_samples = x[random_sample_indices]

        biases = []

        for dilation in self.dilations:
            padding = (self.kernel_size - 1) * dilation // 2
            x_padded = F.pad(x_samples, (padding, padding))
            conv_outputs = F.conv1d(x_padded, self.weights, dilation=dilation, padding=0)

            conv_outputs_flattened = conv_outputs.view(self.num_samples, self.num_kernels, -1)
            quantiles = torch.quantile(conv_outputs_flattened, self.bias_quantiles, dim=2)

            # 重塑为 (num_samples, num_kernels, len(bias_quantiles))
            quantiles = quantiles.permute(1, 2, 0).reshape(self.num_samples, self.num_kernels, len(self.bias_quantiles))

            biases.append(quantiles)

        # 将所有偏置合并为一个张量
        self.biases = torch.cat(biases, dim=1)  # 形状为 (num_samples, num_kernels * len(dilations) * len(bias_quantiles))

    def forward(self, x):
        if self.biases is None:
            self._fit_biases(x)

        batch_size, _, signal_length = x.shape
        features = []

        for dilation in self.dilations:
            padding = (self.kernel_size - 1) * dilation // 2
            x_padded = F.pad(x, (padding, padding))
            conv_output = F.conv1d(x_padded, self.weights, dilation=dilation, padding=0)

            # 对每个卷积核和膨胀因子单独计算特征
            for kernel_idx in range(self.num_kernels):
                for bias_idx in range(len(self.bias_quantiles)):
                    for sample_idx in range(self.num_samples):
                        bias = self.biases[sample_idx, kernel_idx, bias_idx]
                        ppv = (conv_output[:, kernel_idx, :] > bias).float().mean(dim=-1)
                        features.append(ppv)

        return torch.stack(features, dim=1)


In [14]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import f1_score
import time

# 假设你有 HITRocketTransform 类（请确保你引入了该类）
# from your_module import HITRocketTransform

def _quantiles_torch(n: int) -> torch.Tensor:
    phi = (np.sqrt(5) + 1) / 2
    return torch.tensor([(i * phi) % 1 for i in range(1, n + 1)], dtype=torch.float32, device='cuda')


def HITrocket_Liear_Pro(X_train, y_train, X_test, y_test, dataset_name, repeat_times=1):
    X_train = np.nan_to_num(X_train, nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)

    dilations = list(range(1, 16))
    n_quantiles = 10
    bias_quantiles_tensor = _quantiles_torch(n_quantiles)
    bias_quantiles = bias_quantiles_tensor.cpu().tolist()

    num_samples = 5
    results_sum = {}
    total_train_time = 0
    dimension = None

    linear_models = {
        "LinearSVC": LinearSVC(
            C=1.0,
            max_iter=10000,
            class_weight='balanced',
            dual=False,
            tol=1e-4,
            random_state=42
        ),
        "RBF_SVC": SVC(
            C=1.0,
            kernel='rbf',
            gamma='scale',
            class_weight='balanced',
            max_iter=10000,
            random_state=42
        )
    }

    for name in linear_models:
        results_sum[name] = {
            "Train Time (s)": 0.0,
            "Weighted F1": 0.0
        }

    for i in range(repeat_times):
        print(f"🔁 第 {i + 1}/{repeat_times} 次训练")

        start_time = time.time()

        # 转为tensor
        all_segments_tensor_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
        all_segments_tensor_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
        labels_train = torch.tensor(y_train, dtype=torch.long).numpy()
        labels_test = torch.tensor(y_test, dtype=torch.long).numpy()

        # 特征提取
        model = HITRocketTransform(dilations, bias_quantiles, num_samples, hadamard_order=16)
        features_train = model(all_segments_tensor_train).numpy()
        features_test = model(all_segments_tensor_test).numpy()

        # 保存特征
        feat_dir = os.path.join("hitrocket_features", dataset_name)
        os.makedirs(feat_dir, exist_ok=True)
        np.save(os.path.join(feat_dir, "0626features_train.npy"), features_train)
        np.save(os.path.join(feat_dir, "test_features.npy"), features_test)

        # 特征标准化
        scaler = StandardScaler(with_mean=False)
        X_train_feat = scaler.fit_transform(features_train)
        X_test_feat = scaler.transform(features_test)
        dimension = X_train_feat.shape[1]
        print(f"特征维度: {dimension}")

        for name, clf in linear_models.items():
            print(f"    📐 模型: {name}")
            t0 = time.time()
            clf.fit(X_train_feat, labels_train)
            train_time = time.time() - t0

            y_pred = clf.predict(X_test_feat)
            f1 = f1_score(labels_test, y_pred, average='weighted')

            results_sum[name]["Train Time (s)"] += train_time
            results_sum[name]["Weighted F1"] += f1

            # ✅ 保存线性SVM的权重
            if name == "LinearSVC":
                weight_dir = "hitrocket_weights"
                os.makedirs(weight_dir, exist_ok=True)
                np.save(os.path.join(weight_dir, f"{dataset_name}_LinearSVC_weights.npy"), clf.coef_)

        total_train_time += time.time() - start_time

    for name in results_sum:
        results_sum[name]["Train Time (s)"] /= repeat_times
        results_sum[name]["Weighted F1"] /= repeat_times

    total_train_time /= repeat_times

    return results_sum, total_train_time, dimension


def main():
    base_folder = r'F:\datasheet\03时间序列分类\02urc\UCRArchive_2018'
    subfolders = [f for f in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, f))]
    subfolders = subfolders[:2]  # 你可以改成 subfolders[:N]
    results_df = pd.DataFrame(columns=["Dataset"])
    
    cnt = 0

    for subfolder in subfolders:
        train_file = os.path.join(base_folder, subfolder, f"{subfolder}_TRAIN.tsv")
        test_file = os.path.join(base_folder, subfolder, f"{subfolder}_TEST.tsv")

        if os.path.exists(train_file) and os.path.exists(test_file):
            print(f"\n{cnt} - Processing dataset: {subfolder}")
            cnt += 1
            train_data = pd.read_csv(train_file, sep="\t", header=None)
            test_data = pd.read_csv(test_file, sep="\t", header=None)

            X_train = train_data.iloc[:, 1:].values
            y_train = train_data.iloc[:, 0].values
            X_test = test_data.iloc[:, 1:].values
            y_test = test_data.iloc[:, 0].values

            results, total_time, dimension = HITrocket_Liear_Pro(
                X_train, y_train, X_test, y_test, dataset_name=subfolder, repeat_times=1
            )

            row = {"Dataset": subfolder}
            for model_name, metrics in results.items():
                row[f"{model_name} Train Time (s)"] = metrics["Train Time (s)"]
                row[f"{model_name} Weighted F1"] = metrics["Weighted F1"]
            row["Total Time (s)"] = total_time

            results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

    os.makedirs("./00linear_hitrocket_result", exist_ok=True)
    results_df.to_csv(f"./00linear_hitrocket_result/HITrocket_{dimension}_svm_results.csv", index=False)
    print(f"\n✅ Results saved to 'HITrocket_{dimension}_svm_results.csv'.")

    print("\n📊 Final Results:")
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



0 - Processing dataset: ACSF1
🔁 第 1/1 次训练
特征维度: 12000
    📐 模型: LinearSVC
    📐 模型: RBF_SVC

1 - Processing dataset: Adiac
🔁 第 1/1 次训练
特征维度: 12000
    📐 模型: LinearSVC
    📐 模型: RBF_SVC

✅ Results saved to 'HITrocket_12000_svm_results.csv'.

📊 Final Results:
Dataset  LinearSVC Train Time (s)  LinearSVC Weighted F1  RBF_SVC Train Time (s)  RBF_SVC Weighted F1  Total Time (s)
  ACSF1                  0.661783               0.919860                0.296434             0.517166        4.009091
  Adiac                  8.831072               0.808781                4.020482             0.006875       18.047896
